In [1]:
import sys
import os

In [2]:
current_dir = os.getcwd()
root_dir = os.path.abspath(os.path.join(current_dir, '..'))
sys.path.insert(0, root_dir)

In [3]:
import pandas as pd

In [4]:
df_ground_truth = pd.read_csv('./data/ground_truth-new.csv')
ground_truth = df_ground_truth.to_dict(orient="records")

In [5]:
ground_truth[:5]

[{'question': 'Is it too late to start this course if I just found out about it?',
  'document': '74eb249bbf'},
 {'question': 'Do I need to join before a certain date to get a certificate?',
  'document': '74eb249bbf'},
 {'question': "Can I still enroll and complete the course if I'm joining late?",
  'document': '74eb249bbf'},
 {'question': "What's the deadline for submitting my project to earn the certificate?",
  'document': '74eb249bbf'},
 {'question': 'If I start now, will I be able to get certified for this course?',
  'document': '74eb249bbf'}]

In [6]:
from utils.ingest import load_faq_data, build_index

In [7]:
documents = load_faq_data()

In [8]:
documents_llm = [doc for doc in documents if doc['course'] == 'llm-zoomcamp']

In [9]:
documents_llm[:3]

[{'id': '74eb249bbf',
  'course': 'llm-zoomcamp',
  'section': 'General Course-Related Questions',
  'question': 'I just discovered the course. Can I still join?',
  'answer': 'Yes, but if you want to receive a certificate, you need to submit your project while we’re still accepting submissions.'},
 {'id': '977bf7786c',
  'course': 'llm-zoomcamp',
  'section': 'General Course-Related Questions',
  'question': 'Course: I have registered for the LLM Zoomcamp. When can I expect to receive the confirmation email?',
  'answer': "You don't need it. You're accepted. You can also just start learning and submitting homework (while the form is open) without registering. It is not checked against any registered list. Registration is just to gauge interest before the start date."},
 {'id': '489dd1c9d9',
  'course': 'llm-zoomcamp',
  'section': 'General Course-Related Questions',
  'question': 'What is the video/zoom link to the stream for the “Office Hours” or live/workshop sessions?',
  'answer':

In [10]:
documents = documents_llm
index = build_index(documents)

In [11]:
doc_idx = {}
for doc in documents:
    doc_idx[doc['id']] = doc

In [12]:
from dotenv import load_dotenv
from anthropic import Anthropic

In [13]:
load_dotenv()

True

In [14]:
anthropic_client = Anthropic()

In [15]:
from utils.evaluation_utils import RAGWithUsage

In [16]:
assistant = RAGWithUsage(
    index=index,
    llm_client=anthropic_client,
    model="claude-haiku-4-5"
)

In [17]:
rec = ground_truth[0]
question = rec['question']

In [18]:
answer_llm = assistant.rag(question)
print(answer_llm)

# Answer

Yes, it's not too late to start! According to the course information:

**You can join at any time.** The videos and course materials are available whenever you want to start, and you can begin learning immediately.

However, there is one important caveat: **If you want to receive a certificate, you'll need to submit your capstone project while submissions are still being accepted.** The course has deadlines for project submissions, which are listed in the course management platform.

So while you can start the coursework anytime, make sure to be aware of the submission deadline if earning a certificate is important to you. You can check the specific deadlines in the [course management platform](https://courses.datatalks.club/llm-zoomcamp-2026/).


In [19]:
assistant.total_cost()

0.0015

In [20]:
doc_id = rec['document']
original_doc = doc_idx[doc_id]
answer_orig = original_doc['answer']

In [21]:
answer_orig

'Yes, but if you want to receive a certificate, you need to submit your project while we’re still accepting submissions.'

In [22]:
rag_result = {
    "question": question,
    "answer_llm": answer_llm,
    "answer_orig": answer_orig,
    "document": doc_id
}

In [24]:
print(rag_result)

{'question': 'Is it too late to start this course if I just found out about it?', 'answer_llm': "# Answer\n\nYes, it's not too late to start! According to the course information:\n\n**You can join at any time.** The videos and course materials are available whenever you want to start, and you can begin learning immediately.\n\nHowever, there is one important caveat: **If you want to receive a certificate, you'll need to submit your capstone project while submissions are still being accepted.** The course has deadlines for project submissions, which are listed in the course management platform.\n\nSo while you can start the coursework anytime, make sure to be aware of the submission deadline if earning a certificate is important to you. You can check the specific deadlines in the [course management platform](https://courses.datatalks.club/llm-zoomcamp-2026/).", 'answer_orig': 'Yes, but if you want to receive a certificate, you need to submit your project while we’re still accepting subm

## Process all questions

In [25]:
def generate_rag_answer(rec):
    question = rec["question"]
    doc_id = rec["document"]
    original_doc = doc_idx[doc_id]

    answer_llm = assistant.rag(question)
    answer_orig = original_doc["answer"]

    result = {
        "question": question,
        "answer_llm": answer_llm,
        "answer_orig": answer_orig,
        "document": doc_id,
    }

    return result

In [26]:
answer_record = generate_rag_answer(ground_truth[0])
answer_record

{'question': 'Is it too late to start this course if I just found out about it?',
 'answer_llm': "# Answer\n\n**Yes, it's not too late to join!** According to the course information:\n\nYou can start the course whenever you want since the videos and GitHub materials are already available. You don't even need to formally register—you can simply begin learning and submitting homework right away.\n\nHowever, there is one important caveat: **if you want to receive a certificate, you need to submit your capstone project while submissions are still being accepted.** The course has specific deadlines for project submissions that are listed in the [course management platform](https://courses.datatalks.club/llm-zoomcamp-2026/).\n\nSo while you can start learning at any time, make sure to check the current submission deadlines if a certificate is important to you.",
 'answer_orig': 'Yes, but if you want to receive a certificate, you need to submit your project while we’re still accepting submiss

In [28]:
assistant.reset_usage()

In [29]:
assistant.usages

[]

In [30]:
len(ground_truth)

425

In [32]:
from concurrent.futures import ThreadPoolExecutor
from utils.evaluation_utils import map_progress

In [33]:
with ThreadPoolExecutor(max_workers=6) as pool:
    results = map_progress(pool, ground_truth, generate_rag_answer)

  0%|          | 0/425 [00:00<?, ?it/s]

In [34]:
answers = []

for answer_record in results:
    answers.append(answer_record)

In [35]:
assistant.total_cost()

0.830879

In [36]:
df_answers = pd.DataFrame(answers)
df_answers.to_csv("./data/rag-answers-new.csv", index=False)

In [38]:
df_answers.head()

,question,answer_llm,answer_orig,document
0,Is it too late to start this course if I just ...,"# Answer\n\nYes, it's not too late to start th...","Yes, but if you want to receive a certificate,...",74eb249bbf
1,Do I need to join before a certain date to get...,"Based on the context provided, **yes, there is...","Yes, but if you want to receive a certificate,...",74eb249bbf
2,Can I still enroll and complete the course if ...,# Answer: Can I still enroll and complete the ...,"Yes, but if you want to receive a certificate,...",74eb249bbf
3,What's the deadline for submitting my project ...,# Project Submission Deadline\n\nI don't know ...,"Yes, but if you want to receive a certificate,...",74eb249bbf
4,"If I start now, will I be able to get certifie...","Based on the context provided, **whether you c...","Yes, but if you want to receive a certificate,...",74eb249bbf


In [39]:
df_answers.info()

<class 'pandas.DataFrame'>
RangeIndex: 425 entries, 0 to 424
Data columns (total 4 columns):
 #   Column       Non-Null Count  Dtype
---  ------       --------------  -----
 0   question     425 non-null    str  
 1   answer_llm   425 non-null    str  
 2   answer_orig  425 non-null    str  
 3   document     425 non-null    str  
dtypes: str(4)
memory usage: 13.4 KB


In [40]:
answers = df_answers.to_dict(orient="records")

In [41]:
from pydantic import BaseModel, Field
from typing import Literal

In [42]:
class AnswerEvaluation(BaseModel):
    reasoning: str = Field(
        description="Reasoning about the quality of the answer."
    )
    score: Literal["good", "bad"] = Field(
        description="'good' if the answer is correct and complete, 'bad' otherwise."
    )

In [43]:
aqa_judge_instructions = """
You are an expert evaluator. You will be given:
1. A question from a student
2. The original answer from the FAQ (ground truth)
3. An answer generated by an AI assistant

Your task is to decide if the AI answer is semantically equivalent to
the original answer.

Rules:
- The AI answer does NOT need to be word-for-word identical
- It should convey the same key information
- Extra detail is fine as long as the core answer is correct
- Mark 'bad' only if the AI answer is wrong or misses the key point

Be fair and focus on correctness, not style.
""".strip()

In [44]:
aqa_judge_prompt = """
Question:
{question}

Original Answer (ground truth):
{answer_orig}

AI Answer:
{answer_llm}
""".strip()

In [47]:
from utils.evaluation_utils import llm_structured_retry, calc_total_price, calc_price, map_progress

In [45]:
rec = answers[0]

In [46]:
prompt = aqa_judge_prompt.format(
    question=rec['question'],
    answer_orig=rec['answer_orig'],
    answer_llm=rec['answer_llm']
)

In [49]:
eval_result, usage = llm_structured_retry(
    anthropic_client,
    aqa_judge_instructions,
    prompt,
    AnswerEvaluation,
)

In [51]:
eval_result

AnswerEvaluation(reasoning="The AI answer correctly conveys the core message of the original answer: (1) it's not too late to start the course, and (2) if you want a certificate, you need to submit your project while submissions are still being accepted. The AI answer provides additional helpful context about being able to work at your own pace and suggests where to find more information, which are reasonable elaborations. However, the AI answer uses the term 'capstone project' while the original just says 'project' - this is a minor difference that doesn't affect the core correctness. The key information is preserved and accurate.", score='good')

In [52]:
calc_price(usage)

{'input_cost': 0.000695, 'output_cost': 0.000695, 'total_cost': 0.00139}

In [56]:
def evaluate_aqa(question, answer_orig, answer_llm, model="claude-haiku-4-5"):
    prompt = aqa_judge_prompt.format(
        question=question,
        answer_orig=answer_orig,
        answer_llm=answer_llm
    )

    result, usage = llm_structured_retry(
        anthropic_client,
        aqa_judge_instructions,
        prompt,
        AnswerEvaluation,
        model=model,
    )

    return result, usage

In [ ]:
eval_result, usage = evaluate_aqa(
    question=rec["question"],
    answer_orig=rec["answer_orig"],
    answer_llm=rec["answer_llm"]
)

In [ ]:
eval_result

In [57]:
def judge_record(rec):
    eval_result, usage = evaluate_aqa(
        question=rec["question"],
        answer_orig=rec["answer_orig"],
        answer_llm=rec["answer_llm"]
    )

    result = {
        "question": rec["question"],
        "document": rec["document"],
        "score": eval_result.score,
        "reasoning": eval_result.reasoning,
    }

    return result, usage

In [58]:
with ThreadPoolExecutor(max_workers=6) as pool:
    results = map_progress(pool, answers, judge_record)

  0%|          | 0/425 [00:00<?, ?it/s]

In [59]:
evaluations = []
usages = []

for evaluation, usage in results:
    evaluations.append(evaluation)
    usages.append(usage)

In [60]:
df_eval = pd.DataFrame(evaluations)

In [61]:
df_eval.head()

,question,document,score,reasoning
0,Is it too late to start this course if I just ...,74eb249bbf,good,The AI answer conveys the same core message as...
1,Do I need to join before a certain date to get...,74eb249bbf,good,The AI answer provides correct information tha...
2,Can I still enroll and complete the course if ...,74eb249bbf,bad,The AI answer provides more detailed informati...
3,What's the deadline for submitting my project ...,74eb249bbf,good,The original answer states that to receive a c...
4,"If I start now, will I be able to get certifie...",74eb249bbf,bad,The AI answer provides a more detailed and nua...


In [62]:
calc_total_price(usages)

0.7051459999999996

In [63]:
df_eval['score'].value_counts(normalize=True)

score
good    0.877647
bad     0.122353
Name: proportion, dtype: float64

In [64]:
good_count = (df_eval["score"] == "good").sum()
total_count = len(df_eval)
print(f"Good: {good_count}/{total_count} = {good_count/total_count:.2%}")

Good: 373/425 = 87.76%


In [76]:
df_eval[df_eval["score"] == "bad"].iloc[0, 3]

"The AI answer provides more detailed information than the original answer, but let me check if it's semantically equivalent and accurate. The original answer states: 'Yes, but if you want to receive a certificate, you need to submit your project while we're still accepting submissions.' The AI answer confirms this core message in point 2. However, the AI answer adds significant additional information about: (1) being able to follow course content at your own pace, (2) a requirement to complete with a 'live' cohort and participate in peer-reviewing, and (3) a recommendation to check deadlines. The problem is that the original answer does NOT mention these additional requirements about live cohorts or peer-reviewing. The original answer only states the simple rule about submitting the project while submissions are being accepted. The AI answer introduces concepts (live cohort requirement, peer-reviewing requirement) that are NOT in the original answer. While this extra information might

In [66]:
df_eval.to_csv("data/rag-evaluations-new.csv", index=False)